# 6. Classification

Predict the 5 NewsGuard levels (logistic regression with balanced classes; ModernBERT with balanced cross-entropy). Same split, same models for both tasks. Linear models: the penalty is chosen with 3 inner folds of the
training outlets. ModernBERT: 5 epochs, lr 3e-5, effective batch 32, last epoch kept. For every test article the
outputs are saved in `predictions/classification/<model>/`, and every fold model in `Modelli/20_FromScratch/`.

In [ ]:
TASK = "classification"
MODELS = "TF-IDF,Qwen3-8B,Mistral-24B"     # "ModernBERT" runs on the GPU
N_JOBS = 16

In [ ]:
import warnings

import numpy as np
import pandas as pd

import common as C

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 90)
C.OUT.mkdir(parents=True, exist_ok=True)
LOG = C.OUT / "log.txt"
print(C.environment())

import models as MO

art, outlets = MO.load()

In [ ]:
for name in MODELS.split(","):
    for k in range(1, 6):
        if name == "ModernBERT":
            MO.bert_fold(TASK, art, k, "answerdotai/ModernBERT-base", name, LOG)
        else:
            MO.linear_fold(TASK, name, art, k, X=None if name == "TF-IDF" else MO.load_embeddings(name, art),
                           n_jobs=N_JOBS, logfile=LOG)

In [ ]:
for name in MODELS.split(","):
    folder = C.PREDICTIONS / TASK / name
    if name == "ModernBERT":
        print(pd.concat([pd.read_csv(f).assign(fold=f.stem[-1]) for f in sorted(folder.glob("loss_fold*.csv"))])
              .pivot(index="epoch", columns="fold", values="loss").round(4))
    else:
        p = pd.concat([pd.read_csv(f) for f in sorted(folder.glob("penalty_fold*.csv"))])
        print(name, "chosen penalty per fold:", p[p.chosen].set_index("fold").penalty.to_dict())